# Three ways to match document names

Run the cells from top to bottom. The notebook uses only Python's standard library and the CSV beside it. Each method returns a score from 0 to 1. The final cells choose thresholds on a small tuning split, compare the methods on the remaining cases, and save every prediction.

The three methods differ in their **similarity calculation**: trie + edit distance, inverted index + weighted word overlap, and character vectors + cosine similarity. Indexes retrieve candidates; a score and threshold make the pair decision.

## 1. Load the labeled pairs

In [1]:
import csv
import math
import re
from collections import Counter, defaultdict
from functools import lru_cache
from itertools import permutations
from pathlib import Path

data_path = Path("name_matching_test_cases.csv")
if not data_path.exists():
    data_path = Path("outputs/name_matching_test_cases.csv")
with data_path.open(encoding="utf-8-sig", newline="") as file:
    cases = list(csv.DictReader(file))
print(f"Loaded {len(cases)} labeled pairs")

Loaded 200 labeled pairs


## 2. Shared preparation

    Lowercase the text and split it into words. There are no title, relationship, or name-variant rules. The alignment helper pairs each word at most once and allows different word order. Unmatched words score zero.

In [2]:
def words(name):
    return re.findall(r"[a-z]+", name.lower())


def aligned_score(matrix, left_count, right_count):
    size = max(left_count, right_count)
    if size == 0:
        return 0.0
    return max(
        sum(matrix[i][j] for i, j in enumerate(order)
            if i < left_count and j < right_count) / size
        for order in permutations(range(size))
    )

## 3. Approach 1: trie + edit distance

Store the second name's words in a trie. An exact word scores 1. A nearby spelling gets a score based on edit distance. A compatible initial gets only 0.55 because one letter can represent many names.

In [3]:
def build_trie(tokens):
    root = {}
    for index, token in enumerate(tokens):
        node = root
        for letter in token:
            node = node.setdefault(letter, {})
        node.setdefault("_", []).append((index, token))
    return root


def trie_hits(root, query):
    hits = {}
    if len(query) == 1:  # An initial can reach every word with that prefix.
        def collect(node):
            for index, token in node.get("_", []):
                hits[index] = 1.0 if token == query else 0.55
            for letter, child in node.items():
                if letter != "_":
                    collect(child)
        collect(root.get(query, {}))
        return hits

    limit = max(1, len(query) // 3)

    def visit(node, previous):
        for letter, child in node.items():
            if letter == "_":
                continue
            row = [previous[0] + 1]
            for j, wanted in enumerate(query, 1):
                row.append(min(
                    row[-1] + 1, previous[j] + 1,
                    previous[j - 1] + (letter != wanted),
                ))
            for index, token in child.get("_", []):
                if token == query:
                    hits[index] = 1.0
                elif len(token) == 1 and token == query[0]:
                    hits[index] = 0.55
                elif row[-1] <= limit:
                    hits[index] = 1 - row[-1] / max(len(query), len(token))
            if min(row) <= limit:
                visit(child, row)

    visit(root, list(range(len(query) + 1)))
    return hits


def trie_score(case):
    left, right = words(case["name_1"]), words(case["name_2"])
    root = build_trie(right)
    matrix = []
    for token in left:
        hits = trie_hits(root, token)
        matrix.append([hits.get(j, 0.0) for j in range(len(right))])
    return aligned_score(matrix, len(left), len(right))

## 4. Approach 2: inverted index + weighted overlap

Each exact word points to the IDs of documents containing it. A candidate needs at least one shared word. Weighted Jaccard scores shared words against **all** words in both names, so an extra middle name reduces the score. Words appearing in many documents get less weight.

In [4]:
postings = defaultdict(set)
document_frequency = Counter()
for case in cases:
    for name in (case["name_1"], case["name_2"]):
        document_frequency.update(set(words(name)))
    for token in set(words(case["name_2"])):
        postings[token].add(case["doc_2_id"])

document_count = 2 * len(cases)
word_weight = {
    token: 1 + math.log((document_count + 1) / (count + 1))
    for token, count in document_frequency.items()
}


def inverted_score(case):
    left, right = set(words(case["name_1"])), set(words(case["name_2"]))
    candidates = set().union(*(postings[token] for token in left)) if left else set()
    if case["doc_2_id"] not in candidates:
        return 0.0
    shared = sum(word_weight[token] for token in left & right)
    all_words = sum(word_weight[token] for token in left | right)
    return shared / all_words if all_words else 0.0

## 5. Approach 3: character vectors + cosine similarity

Represent each word by its two- and three-character fragments. Compare word vectors with cosine similarity, then take the best one-to-one alignment. This runs in memory; a vector database would be useful only for searching a large collection.

In [5]:
@lru_cache(maxsize=None)
def vector(word):
    padded = f"^{word}$"
    return Counter(
        padded[i:i + width]
        for width in (2, 3)
        for i in range(len(padded) - width + 1)
    )


def cosine(a, b):
    va, vb = vector(a), vector(b)
    dot = sum(count * vb.get(fragment, 0) for fragment, count in va.items())
    norm_a = math.sqrt(sum(count * count for count in va.values()))
    norm_b = math.sqrt(sum(count * count for count in vb.values()))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


def vector_score(case):
    left, right = words(case["name_1"]), words(case["name_2"])
    matrix = [[cosine(a, b) for b in right] for a in left]
    return aligned_score(matrix, len(left), len(right))

## 6. Compare all three methods

Use every fifth pair to select a threshold. Report results on the other pairs. The illustrative cost is **5 per false match** and **1 per missed match**, because an incorrect automatic identity match is especially harmful. Ties choose the higher threshold.

In [6]:
methods = {"trie": trie_score, "inverted": inverted_score, "vector": vector_score}
scored = [{**case, **{name: fn(case) for name, fn in methods.items()}}
          for case in cases]
tuning = [case for i, case in enumerate(scored) if i % 5 == 0]
evaluation = [case for i, case in enumerate(scored) if i % 5 != 0]


def counts(rows, method, threshold):
    tp = fp = fn = tn = 0
    for case in rows:
        predicted = case[method] >= threshold
        actual = case["label"] == "match"
        if predicted and actual: tp += 1
        elif predicted: fp += 1
        elif actual: fn += 1
        else: tn += 1
    return tp, fp, fn, tn


def tuning_cost(method, cutoff):
    _, fp, fn, _ = counts(tuning, method, cutoff)
    return 5 * fp + fn, -cutoff


thresholds = {}
print("method     cutoff  TP  FP  FN  TN  precision  recall  cost")
for method in methods:
    thresholds[method] = min(
        (step / 100 for step in range(101)),
        key=lambda cutoff: tuning_cost(method, cutoff),
    )
    tp, fp, fn, tn = counts(evaluation, method, thresholds[method])
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    print(f"{method:10} {thresholds[method]:.2f}   {tp:2}  {fp:2}  {fn:2}  {tn:2}"
          f"    {precision:.2f}      {recall:.2f}    {5*fp+fn}")

method     cutoff  TP  FP  FN  TN  precision  recall  cost
trie       0.94   14   0  71  75    1.00      0.16    71
inverted   1.00   11   0  74  75    1.00      0.13    74
vector     1.00   10   0  75  75    1.00      0.12    75


## 7. Save every score and prediction

The output CSV includes the tuning/evaluation split, true label, three scores, and three decisions. This makes individual errors inspectable.

In [7]:
result_path = data_path.with_name("name_matching_predictions.csv")
fields = ["case_id", "name_1", "name_2", "label", "category", "split"]
fields += [f"{method}_score" for method in methods]
fields += [f"{method}_prediction" for method in methods]
with result_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    for i, case in enumerate(scored):
        row = {field: case[field] for field in fields[:5]}
        row["split"] = "tuning" if i % 5 == 0 else "evaluation"
        for method in methods:
            row[f"{method}_score"] = round(case[method], 4)
            row[f"{method}_prediction"] = (
                "match" if case[method] >= thresholds[method] else "non-match"
            )
        writer.writerow(row)
print(f"Saved {result_path}")

Saved name_matching_predictions.csv


### Interpretation

The dataset is synthetic, and some initials or omitted-name pairs are ambiguous from text alone. A high score means similar name strings, not verified identity. This exercise uses binary decisions; a real KYC system should send uncertain cases to review and validate thresholds on identity-verified records.